In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST, FashionMNIST
from torchvision.transforms import v2

from models import LeNet5, FberNet

torch.manual_seed(10406308320747545401);

In [ ]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(dtype=torch.float32), v2.Normalize(mean=[0], std=[1])])

train_dataset = MNIST(root='data/', train=True, transform=transform, download=True)
test_dataset = MNIST(root='data/', train=False, transform=transform, download=True)
#train_dataset = FashionMNIST(root='data/', train=True, transform=transform, download=True)
#test_dataset = FashionMNIST(root='data/', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
model = FberNet()

num_parameters = sum([p.numel() for p in model.parameters() if p.requires_grad])
print(f'Model has {num_parameters} parameters.')

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 10

for epoch in range(epochs):

    running_loss = 0.0
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    running_loss /= len(train_dataset)

    hits = 0
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predictions = torch.max(outputs.data, 1)
            hits += (predictions == labels).sum().item()

    accuracy = 100 * hits / len(test_dataset)
    
    print(f'[Epoch {epoch + 1:2d}/{epochs}] Training loss: {running_loss:.6f} - Test accuracy: {accuracy:.2f} %')

In [ ]:
#torch.save(model.state_dict(), 'lenet5_fashion_mnist.pth')
#torch.save(model.state_dict(), 'lenet5_mnist.pth')
torch.save(model.state_dict(), 'fbernet_mnist.pth')